# opndet — Colab Quickstart

Train a tiny single-class detector on your COCO dataset. Runtime: **T4/L4/A100** (Runtime → Change runtime type → GPU).

What this notebook does:
1. Installs opndet from GitHub.
2. Mounts Google Drive (for dataset + checkpoints).
3. Generates a training config and edits it.
4. Trains, exports ONNX, downloads artifacts.

## 1. Install opndet

Repo: `bherbruck/opndet`. Override `OPNDET_REPO` if you fork.

In [ ]:
OPNDET_REPO = "bherbruck/opndet"   # e.g. "bherbruck/opndet"
BRANCH      = "main"

!pip install -q --upgrade pip
!pip install -q git+https://github.com/{OPNDET_REPO}.git@{BRANCH}
!opndet info

## 2. Verify GPU

If this prints `cpu` you didn't enable a GPU runtime.

In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    !nvidia-smi -q -d MEMORY | grep -A 2 'FB Memory Usage'

## 3. Mount Drive

Recommended layout in your Drive:
```
MyDrive/opndet/
  dataset/<your-data>/      # images + COCO json(s)
  runs/                      # checkpoints (created automatically)
```

Skip this cell if you'd rather upload the dataset zip directly to Colab (next cell).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/opndet/runs
!ls /content/drive/MyDrive/opndet

## 3b. (Alt) Upload dataset zip directly

Use this if the dataset isn't in Drive. Pick a zip with images + COCO JSON inside.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick your dataset.zip
import zipfile, os
for fn in uploaded:
    with zipfile.ZipFile(fn) as z:
        z.extractall('/content/dataset')
    os.remove(fn)
!ls /content/dataset

## 4. Generate training config

Dumps the bundled template. Adjust paths in the next cell.

In [ ]:
!opndet init-config --out /content/train.yaml
!cat /content/train.yaml

## 5. Edit the config

Set `data.sources` to your COCO JSON(s) + image dir(s), and `out_dir` to a Drive path so checkpoints survive session disconnects.

In [ ]:
TRAIN_YAML = """
out_dir: /content/drive/MyDrive/opndet/runs/exp1
seed: 0
device: cuda
amp: true

model_config: bbox-s          # bbox-n | bbox-s | bbox-m
model:
  stride: 4

data:
  sources:
    # EDIT: point at your dataset(s)
    - {coco: /content/dataset/_annotations.coco.json, images: /content/dataset}
    # add more here:
    # - {coco: /content/drive/MyDrive/opndet/dataset/site_b/anno.json, images: /content/drive/MyDrive/opndet/dataset/site_b}
  split_ratios: [0.8, 0.1, 0.1]

batch_size: 32
num_workers: 2
epochs: 100
lr: 3.0e-3
weight_decay: 1.0e-4
warmup_steps: 200
eval_threshold: 0.3

augment:
  enabled: true
  brightness: 0.4
  contrast: 0.4
  gamma: [0.6, 1.6]
  hue: 20
  saturation: 0.5
  grayscale_prob: 0.2
  blur_prob: 0.1
  noise_sigma: 0.02
  hflip_prob: 0.5
  vflip_prob: 0.5
  rotate90_prob: 0.5
  scale_jitter: [0.7, 1.3]
  translate_frac: 0.1

loss:
  w_hm: 1.0
  w_cxy: 1.0
  w_wh: 5.0
  focal_alpha: 2.0
  focal_beta: 4.0
"""
with open('/content/train.yaml', 'w') as f:
    f.write(TRAIN_YAML)
print(open('/content/train.yaml').read())

## 6a. Launch TensorBoard

Live scalars (loss, P/R/F1, LR) and per-epoch image grids of validation predictions (GT in **red**, preds in **green**). Refresh the TB pane every epoch to see new images.


In [ ]:
%load_ext tensorboard
OUT_DIR = '/content/drive/MyDrive/opndet/runs/exp1'
%tensorboard --logdir {OUT_DIR}/tb


## 6. Train

Logs print per epoch: loss + precision/recall/F1 on val. Best ckpt saved to `out_dir/best.pt`.

In [ ]:
!opndet train --config /content/train.yaml

## 7. Export ONNX (opset 13)

Validates allowed-op list and runs PyTorch ↔ ORT parity check.

In [ ]:
OUT_DIR = '/content/drive/MyDrive/opndet/runs/exp1'
!opndet export --model bbox-s --ckpt {OUT_DIR}/best.pt --out {OUT_DIR}/opndet.onnx

## 8. Quick visual check

Predict on one of your training images, render a vis.

In [ ]:
import glob, os
imgs = sorted(glob.glob('/content/dataset/*.jpg'))[:1] or sorted(glob.glob('/content/drive/MyDrive/opndet/dataset/**/*.jpg', recursive=True))[:1]
if imgs:
    test_img = imgs[0]
    !opndet predict --image "{test_img}" --model bbox-s --ckpt {OUT_DIR}/best.pt --threshold 0.3 --device cuda --save /content/vis.jpg
    from IPython.display import Image
    display(Image('/content/vis.jpg'))
else:
    print('no images found — set test_img manually')

## 9. Download artifacts

If `out_dir` was on Drive, you already have everything. Otherwise:

In [ ]:
from google.colab import files
files.download(f'{OUT_DIR}/best.pt')
files.download(f'{OUT_DIR}/opndet.onnx')

---

## Tips

- **Session disconnects.** Colab kills idle sessions. Set `out_dir` to Drive so checkpoints survive. Restart, re-run cells 1–4, then re-launch training pointing at the same `out_dir` — best.pt will be reloaded if you wire it (TODO: add `--resume` flag).
- **OOM.** Drop `batch_size` (32 → 16 or 8). The bbox-s model fits comfortably on a T4.
- **Faster iteration.** Start with `bbox-n` (0.31M params). Once your data + aug pipeline is solid, scale to `bbox-s` or `bbox-m`.
- **Multi-source datasets.** Add more entries under `data.sources`. They merge into one pool before splitting.